# vowl · Advanced Usage

Advanced ways to feed data into **vowl**. This notebook builds on the [Basic Tutorial](../1_basic_tutorial/basic_tutorial.ipynb): start there for setup, auto-mapped validation, and working with results. Run it top-to-bottom.

Where the Basic Tutorial lets vowl auto-detect your input, this notebook takes explicit control of the connection and narrows *which rows* get validated:

## Contents

1. [Setup](#1-setup)
2. [Explicitly Defined Adapter](#2-explicitly-defined-adapter): incl. `PooledAdapter`
3. [Filtering Rows Before Validation](#3-filtering-rows-before-validation)

Generated CSV/JSON artifacts (if any) are written to this notebook's local `outputs/` folder.


<a id="1-setup"></a>
## 1. Setup

Install the dependencies (skipped during execution), then resolve the dataset paths and shared imports used throughout the notebook. This mirrors the Basic Tutorial's setup so the notebook runs on its own.


In [1]:
# Install vowl (skipped during execution)
# %pip install 'vowl[all]'

In [2]:
from pathlib import Path

# Walk up to the repo root (works no matter how deep this notebook sits)
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "tests" / "hdb_resale").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# HDB Resale dataset (used throughout this notebook)
HDB_DIR = REPO_ROOT / "tests" / "hdb_resale"
HDB_CSV = HDB_DIR / "HDBResaleWithErrors.csv"
HDB_CONTRACT = HDB_DIR / "hdb_resale_simple.yaml"  # Switch to "hdb_resale.yaml" for a more complex contract

# Common imports used throughout this notebook
import pandas as pd
from vowl import validate_data

# --- Quieten known, benign warnings emitted by this demo ---
import warnings

# vowl surfaces UserWarnings (Arrow type coercion, multi-schema adapter reuse)
# that are informational for this demo dataset.
warnings.filterwarnings("ignore", category=UserWarning,
                        module=r"vowl\.validation\.runner")

<a id="2-explicitly-defined-adapter"></a>
## 2. Explicitly Defined Adapter

Instead of relying on auto-mapping (see the [Basic Tutorial](../1_basic_tutorial/basic_tutorial.ipynb#2-auto-mapped-validation)), you can explicitly create an `IbisAdapter` and pass it to `validate_data(adapter=...)`. This gives you full control over the connection, backend, and configuration.

Here we use DuckDB in-memory by materialising the table before validating. The SQL checks run **client-side** within the DuckDB process. This same pattern works with any of the [20+ Ibis backends](https://github.com/ibis-project/ibis) (PostgreSQL, Snowflake, BigQuery, etc.).


In [3]:
import ibis
from vowl import validate_data
from vowl.adapters import IbisAdapter

# Create an in-memory DuckDB connection and register the data
con = ibis.duckdb.connect()
hdb_df = pd.read_csv(HDB_CSV, low_memory=False).fillna("")
hdb_df["lease_commence_date"] = hdb_df["lease_commence_date"].astype(str)
con.create_table("hdb_resale_prices", hdb_df)

# Validate using the IbisAdapter. SQL checks run client-side in DuckDB
adapter = IbisAdapter(con)
result = validate_data(contract=str(HDB_CONTRACT), adapter=adapter)

In [4]:
# Show the full validation report
result.display_full_report()



=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           c11443ee-542f-4442-b28d-2d224342be37
   Schemas:               hdb_resale_prices

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       16 / 20 (80.0%)

   hdb_resale_prices:
     Overall:
       Checks Pass Rate:       16 / 20 (80.0%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       16 / 20 (80.0%)
       ERRORED Checks:         0
       Unique Passed Rows:     201,861 / 201,879 (99.9%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)
       ERRORED Checks:         0
       Non-unique Failed Rows: 0


 CHECK RESULTS
+-----------------------------------------+---------------------------------------+-------------------+--------+---------------+---------------+--------+----------------+
| check_id                                | Target                                | tables_in_query   | status | operator      | expected      | actua

ValidationResult(passed=False, checks=20, passed_checks=16, failed_checks=4)

<a id="21-pooled-adapter"></a>
### 2.1 Running Checks Concurrently with `PooledAdapter`

By default an adapter runs each check sequentially on a single connection. When a contract has many independent checks and the backend can serve several queries at once, you can run them **concurrently** by wrapping a connection *factory* in a `PooledAdapter`.

`PooledAdapter` keeps a thread-safe pool of adapter instances (one connection per worker, each used by at most one thread at a time) and dispatches checks across them. You give it:

- `factory`: a zero-argument callable that returns a fresh `BaseAdapter` (e.g. an `IbisAdapter` over a new connection). It is called once per pooled connection, so any table the contract references must be available on every connection the factory builds.
- `max_concurrency`: the maximum number of connections (and therefore in-flight checks) to use.

Because `PooledAdapter` is a connection pool rather than a single Ibis connection, it is wired in through a `MultiSourceAdapter` (the same entry point used for multi-source validation in the [Multiple Sources notebook](../2_multiple_sources/multiple_sources.ipynb)) and passed via `validate_data(adapters=...)`. The verdicts are identical to a sequential run; pooling only changes how the checks are scheduled, not their results.


In [5]:
from vowl.adapters import PooledAdapter, MultiSourceAdapter

# A factory that builds a *fresh* adapter (new connection) on each call.
# The pool calls it once per worker, so the table must be registered on
# every connection it creates -- hence we load the CSV inside the factory.
def make_duckdb_adapter():
    con = ibis.duckdb.connect()
    df = pd.read_csv(HDB_CSV, low_memory=False).fillna("")
    df["lease_commence_date"] = df["lease_commence_date"].astype(str)
    con.create_table("hdb_resale_prices", df)
    return IbisAdapter(con)

# Wrap the factory in a pool, then route it through a MultiSourceAdapter
# keyed by the schema name the contract validates.
pooled = PooledAdapter(factory=make_duckdb_adapter, max_concurrency=4)
multi = MultiSourceAdapter({"hdb_resale_prices": pooled})

result = validate_data(contract=str(HDB_CONTRACT), adapters=multi)

print(f"Ran {result.summary['validation_summary']['total_checks']} checks "
      f"across up to {pooled.max_concurrency} concurrent connections.")

Ran 20 checks across up to 4 concurrent connections.


In [6]:
# Same ValidationResult as a sequential run -- just scheduled concurrently
result.print_summary()



=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           c11443ee-542f-4442-b28d-2d224342be37
   Schemas:               hdb_resale_prices

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       16 / 20 (80.0%)

   hdb_resale_prices:
     Overall:
       Checks Pass Rate:       16 / 20 (80.0%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       16 / 20 (80.0%)
       ERRORED Checks:         0
       Unique Passed Rows:     201,861 / 201,879 (99.9%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)
       ERRORED Checks:         0
       Non-unique Failed Rows: 0


 CHECK RESULTS
+-----------------------------------------+---------------------------------------+-------------------+--------+---------------+---------------+--------+----------------+
| check_id                                | Target                                | tables_in_query   | status | operator      | expected      | actua

ValidationResult(passed=False, checks=20, passed_checks=16, failed_checks=4)

<a id="3-filtering-rows"></a>
## 3. Filtering Rows Before Validation

Filter conditions let you validate only a subset of your data, useful for incremental validation of new records.

Table name keys in `filter_conditions` support **wildcard patterns** via Python's [`fnmatch`](https://docs.python.org/3/library/fnmatch.html) module:

| Pattern | Matches | Does Not Match |
|---------|---------|----------------|
| `"emp*"` | `employees`, `emp_history`, `emp_details` | `department` |
| `"table?"` | `table1`, `table2` | `table10` |
| `"[abc]*"` | `accounts`, `billing`, `customers` | `delivery` |
| `"*_archive"` | `orders_archive`, `customers_archive` | `orders` |
| `"*"` | All tables (useful for tenant filtering) | -- |

If multiple patterns match a table, all conditions are combined with AND.

The three examples below share one DuckDB connection.


In [7]:
from vowl.adapters import IbisAdapter, FilterCondition

con = ibis.duckdb.connect()
hdb_df = pd.read_csv(HDB_CSV, low_memory=False).fillna("")
hdb_df["lease_commence_date"] = hdb_df["lease_commence_date"].astype(str)
con.create_table("hdb_resale_prices", hdb_df)

# Dict-style filter: only validate rows from 2024 onwards
adapter = IbisAdapter(
    con,
    filter_conditions={
        "hdb_resale_prices": {
            "field": "month",
            "operator": ">=",
            "value": "2024-01",
        }
    },
)

result = validate_data(contract=str(HDB_CONTRACT), adapter=adapter)

In [8]:
# Dict-style filter (rows from 2024 onwards)
result.print_summary()



=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           c11443ee-542f-4442-b28d-2d224342be37
   Schemas:               hdb_resale_prices

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       19 / 20 (95.0%)

   hdb_resale_prices:
     Overall:
       Checks Pass Rate:       19 / 20 (95.0%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       19 / 20 (95.0%)
       ERRORED Checks:         0
       Unique Passed Rows:     32,727 / 32,729 (99.9%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)
       ERRORED Checks:         0
       Non-unique Failed Rows: 0


 CHECK RESULTS
+-----------------------------------------+---------------------------------------+-------------------+--------+---------------+---------------+--------+----------------+
| check_id                                | Target                                | tables_in_query   | status | operator      | expected      | actual 

ValidationResult(passed=False, checks=20, passed_checks=19, failed_checks=1)

In [9]:
# Multiple filter conditions on the same table (combined with AND)
adapter = IbisAdapter(
    con,
    filter_conditions={
        "hdb_resale_prices": [
            FilterCondition(field="month", operator=">=", value="2024-01"),
            FilterCondition(field="town", operator="=", value="ANG MO KIO"),
        ]
    },
)

result = validate_data(contract=str(HDB_CONTRACT), adapter=adapter)

In [10]:
# Multiple conditions combined with AND
result.print_summary()



=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           c11443ee-542f-4442-b28d-2d224342be37
   Schemas:               hdb_resale_prices

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       20 / 20 (100.0%)

   hdb_resale_prices:
     Overall:
       Checks Pass Rate:       20 / 20 (100.0%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       20 / 20 (100.0%)
       ERRORED Checks:         0
       Unique Passed Rows:     1,264 / 1,264 (100.0%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)
       ERRORED Checks:         0
       Non-unique Failed Rows: 0


 CHECK RESULTS
+-----------------------------------------+---------------------------------------+-------------------+--------+---------------+---------------+--------+----------------+
| check_id                                | Target                                | tables_in_query   | status | operator      | expected      | actua

ValidationResult(passed=True, checks=20, passed_checks=20, failed_checks=0)

In [11]:
# Wildcard pattern matching on table names
# The "*" pattern applies a filter to ALL tables in the contract
adapter = IbisAdapter(
    con,
    filter_conditions={
        # "*" matches every table name (via fnmatch). Useful for multi-tenant
        # scenarios where every table has a shared column like tenant_id.
        "*": FilterCondition(field="month", operator=">=", value="2020-01"),
    },
)

result = validate_data(contract=str(HDB_CONTRACT), adapter=adapter)

In [12]:
# Wildcard "*" filter applied to every table
result.print_summary()



=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           c11443ee-542f-4442-b28d-2d224342be37
   Schemas:               hdb_resale_prices

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       19 / 20 (95.0%)

   hdb_resale_prices:
     Overall:
       Checks Pass Rate:       19 / 20 (95.0%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       19 / 20 (95.0%)
       ERRORED Checks:         0
       Unique Passed Rows:     137,616 / 137,623 (99.9%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)
       ERRORED Checks:         0
       Non-unique Failed Rows: 0


 CHECK RESULTS
+-----------------------------------------+---------------------------------------+-------------------+--------+---------------+---------------+--------+----------------+
| check_id                                | Target                                | tables_in_query   | status | operator      | expected      | actua

ValidationResult(passed=False, checks=20, passed_checks=19, failed_checks=1)